[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/10-agentic-ai/notebooks/01-tools-and-function-calling.ipynb)

# Notebook 01 — Tools and function calling

**Goal:** understand exactly what "tool use" (also called **function calling**) is, by building every piece by hand:

1. a normal Python function (the **tool**),
2. a **JSON Schema** that describes its inputs,
3. the **OpenAI** and **Anthropic** wrappers around that schema, side by side,
4. the code that reads a model's **tool call**, checks it, runs the function and sends the **tool result** back.

Everything runs **offline** with no API key. Where a real API call is shown, it only runs if you have set a key.

*Jargon:* **JSON** is a text format for data that looks like Python dicts and lists. **JSON Schema** is a standard way to say what shape a piece of JSON must have.

## Step 1 — A normal Python function

A tool is just a function. The model will never run it. It can only *ask* us to run it.

In [1]:
def get_weather(city: str, unit: str = "celsius") -> dict:
    """Pretend weather service (a real one would call a weather API)."""
    fake_data = {"taipei": 24, "hyderabad": 31, "london": 9}
    temp_c = fake_data.get(city.lower(), 20)
    temp = temp_c if unit == "celsius" else round(temp_c * 9 / 5 + 32)
    return {"city": city, "temperature": temp, "unit": unit}

print(get_weather("Taipei"))
print(get_weather("London", unit="fahrenheit"))

{'city': 'Taipei', 'temperature': 24, 'unit': 'celsius'}
{'city': 'London', 'temperature': 48, 'unit': 'fahrenheit'}


## Step 2 — Describe the inputs with JSON Schema

The model cannot read our Python code. We describe the function's inputs in JSON Schema. Read it as: *"an object with a text field `city` (required) and a field `unit` that must be one of two words."*

In [2]:
import json

WEATHER_PARAMS = {
    "type": "object",
    "properties": {
        "city": {"type": "string", "description": "City name, e.g. Taipei"},
        "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "Temperature unit"},
    },
    "required": ["city"],
}
print(json.dumps(WEATHER_PARAMS, indent=2))

{
  "type": "object",
  "properties": {
    "city": {
      "type": "string",
      "description": "City name, e.g. Taipei"
    },
    "unit": {
      "type": "string",
      "enum": [
        "celsius",
        "fahrenheit"
      ],
      "description": "Temperature unit"
    }
  },
  "required": [
    "city"
  ]
}


## Step 3 — The same tool, wrapped for OpenAI and for Anthropic

Both companies accept the same schema. Only the wrapper differs:

* **OpenAI** puts it under `function.parameters`.
* **Anthropic** puts it under `input_schema`.

In [3]:
OPENAI_TOOL = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current temperature for a city. Use when the user asks about weather.",
        "parameters": WEATHER_PARAMS,
    },
}
ANTHROPIC_TOOL = {
    "name": "get_weather",
    "description": "Get the current temperature for a city. Use when the user asks about weather.",
    "input_schema": WEATHER_PARAMS,
}
print("OpenAI:\n", json.dumps(OPENAI_TOOL, indent=2))
print("\nAnthropic:\n", json.dumps(ANTHROPIC_TOOL, indent=2))

OpenAI:
 {
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get the current temperature for a city. Use when the user asks about weather.",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "City name, e.g. Taipei"
        },
        "unit": {
          "type": "string",
          "enum": [
            "celsius",
            "fahrenheit"
          ],
          "description": "Temperature unit"
        }
      },
      "required": [
        "city"
      ]
    }
  }
}

Anthropic:
 {
  "name": "get_weather",
  "description": "Get the current temperature for a city. Use when the user asks about weather.",
  "input_schema": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string",
        "description": "City name, e.g. Taipei"
      },
      "unit": {
        "type": "string",
        "enum": [
          "celsius",
          "fahrenheit"
   

Because the formats are so close, converting between them is a three-line function. Frameworks such as LangChain do this for you.

In [4]:
def openai_to_anthropic(tool: dict) -> dict:
    f = tool["function"]
    return {"name": f["name"], "description": f["description"], "input_schema": f["parameters"]}

print(openai_to_anthropic(OPENAI_TOOL) == ANTHROPIC_TOOL)

True


## Step 4 — Let Python write the schema for you

Writing schemas by hand is boring and error-prone. Libraries read the function's **type hints** (the `: str` parts) and **docstring** and build the schema automatically. Here is a tiny home-made version using Python's `inspect` module, so you can see there is no magic.

In [5]:
import inspect

PY_TO_JSON = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}

def schema_from_function(fn) -> dict:
    sig = inspect.signature(fn)                      # the function's parameters
    props, required = {}, []
    for name, p in sig.parameters.items():
        props[name] = {"type": PY_TO_JSON.get(p.annotation, "string")}
        if p.default is inspect.Parameter.empty:     # no default value -> required
            required.append(name)
    return {"name": fn.__name__,
            "description": (fn.__doc__ or "").strip(),
            "input_schema": {"type": "object", "properties": props, "required": required}}

print(json.dumps(schema_from_function(get_weather), indent=2))

{
  "name": "get_weather",
  "description": "Pretend weather service (a real one would call a weather API).",
  "input_schema": {
    "type": "object",
    "properties": {
      "city": {
        "type": "string"
      },
      "unit": {
        "type": "string"
      }
    },
    "required": [
      "city"
    ]
  }
}


The real libraries do the same with more care. LangChain's `@tool` decorator (installed here) is one example:

In [6]:
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

@tool
def get_weather_lc(city: str, unit: str = "celsius") -> str:
    """Get the current temperature for a city."""
    return json.dumps(get_weather(city, unit))

print(json.dumps(convert_to_openai_tool(get_weather_lc), indent=2))

{
  "type": "function",
  "function": {
    "name": "get_weather_lc",
    "description": "Get the current temperature for a city.",
    "parameters": {
      "properties": {
        "city": {
          "type": "string"
        },
        "unit": {
          "default": "celsius",
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "type": "object"
    }
  }
}


## Step 5 — What the model sends back (offline simulation)

When a model decides to use a tool, it does **not** run it. It replies with a structured request. The two providers shape that reply differently:

* OpenAI: `tool_calls[i].function.arguments` is a JSON **string** — you must `json.loads` it.
* Anthropic: a content block of `type == "tool_use"` whose `input` is already a **dict**.

Below we write those replies by hand (exactly the shape the real APIs return, simplified).

In [7]:
openai_reply = {"role": "assistant", "content": None, "tool_calls": [
    {"id": "call_abc", "type": "function",
     "function": {"name": "get_weather", "arguments": '{"city": "Taipei"}'}}]}

anthropic_reply = {"role": "assistant", "stop_reason": "tool_use", "content": [
    {"type": "text", "text": "Let me check."},
    {"type": "tool_use", "id": "toolu_123", "name": "get_weather", "input": {"city": "Taipei"}}]}

print(type(openai_reply["tool_calls"][0]["function"]["arguments"]))   # str
print(type(anthropic_reply["content"][1]["input"]))                    # dict

<class 'str'>
<class 'dict'>


## Step 6 — Validate before you run

The model can make mistakes: a missing field, a wrong type, a value not in the `enum`. **Never trust tool arguments blindly.** The `jsonschema` library checks a dict against a schema.

In [8]:
from jsonschema import validate, ValidationError

def check_args(args: dict, schema: dict) -> str | None:
    try:
        validate(instance=args, schema=schema)
        return None                         # None = no problem
    except ValidationError as e:
        return f"Invalid arguments: {e.message}"

print(check_args({"city": "Taipei"}, WEATHER_PARAMS))
print(check_args({"unit": "kelvin"}, WEATHER_PARAMS))

None
Invalid arguments: 'city' is a required property


## Step 7 — Dispatch: from tool call to tool result

A **registry** maps tool names to Python functions. The **dispatcher** looks up the name, validates, runs, and wraps the result in the format the provider expects. Errors are *returned to the model as text* so it can try again — a crash would end the conversation.

In [9]:
TOOLS = {"get_weather": (get_weather, WEATHER_PARAMS)}

def run_tool(name: str, args: dict) -> str:
    if name not in TOOLS:
        return f"Error: unknown tool '{name}'. Available: {list(TOOLS)}"
    fn, schema = TOOLS[name]
    problem = check_args(args, schema)
    if problem:
        return problem
    return json.dumps(fn(**args))

# OpenAI style result message
tc = openai_reply["tool_calls"][0]
openai_result = {"role": "tool", "tool_call_id": tc["id"],
                 "content": run_tool(tc["function"]["name"], json.loads(tc["function"]["arguments"]))}
# Anthropic style result message (goes in a *user* turn)
block = anthropic_reply["content"][1]
anthropic_result = {"role": "user", "content": [
    {"type": "tool_result", "tool_use_id": block["id"], "content": run_tool(block["name"], block["input"])}]}

print(openai_result)
print(anthropic_result)
print(run_tool("get_stock_price", {"ticker": "TSM"}))

{'role': 'tool', 'tool_call_id': 'call_abc', 'content': '{"city": "Taipei", "temperature": 24, "unit": "celsius"}'}
{'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_123', 'content': '{"city": "Taipei", "temperature": 24, "unit": "celsius"}'}]}
Error: unknown tool 'get_stock_price'. Available: ['get_weather']


## Step 8 — The real calls (only run with a key)

This is the real code for both providers. Without a key it prints a message and moves on, so the notebook always runs. To try it, set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` (in Colab: the key icon → Secrets) and `pip install openai anthropic`.

Model names change often — as of late 2026, check the official model pages and set `OPENAI_MODEL` / `ANTHROPIC_MODEL`.

In [10]:
import os

question = "What's the weather in Taipei?"

if os.getenv("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    model = os.getenv("OPENAI_MODEL", "gpt-5-mini")
    msgs = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model=model, messages=msgs, tools=[OPENAI_TOOL])
    m = r.choices[0].message
    if m.tool_calls:
        msgs.append(m)
        for t in m.tool_calls:
            msgs.append({"role": "tool", "tool_call_id": t.id,
                         "content": run_tool(t.function.name, json.loads(t.function.arguments))})
        r = client.chat.completions.create(model=model, messages=msgs, tools=[OPENAI_TOOL])
    print("OpenAI:", r.choices[0].message.content)
else:
    print("No OPENAI_API_KEY - skipping the real OpenAI call (offline simulation above shows the shapes).")

if os.getenv("ANTHROPIC_API_KEY"):
    import anthropic
    client = anthropic.Anthropic()
    model = os.getenv("ANTHROPIC_MODEL", "claude-opus-5-5")
    msgs = [{"role": "user", "content": question}]
    r = client.messages.create(model=model, max_tokens=400, tools=[ANTHROPIC_TOOL], messages=msgs)
    if r.stop_reason == "tool_use":
        msgs.append({"role": "assistant", "content": r.content})
        msgs.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": b.id, "content": run_tool(b.name, b.input)}
            for b in r.content if b.type == "tool_use"]})
        r = client.messages.create(model=model, max_tokens=400, tools=[ANTHROPIC_TOOL], messages=msgs)
    print("Claude:", r.content[0].text)
else:
    print("No ANTHROPIC_API_KEY - skipping the real Anthropic call.")

No OPENAI_API_KEY - skipping the real OpenAI call (offline simulation above shows the shapes).
No ANTHROPIC_API_KEY - skipping the real Anthropic call.


## Recap

* A tool = **name + description + JSON Schema** for the inputs.
* OpenAI wraps it as `function.parameters`; Anthropic as `input_schema`.
* The model only **requests** a call. Your code validates, runs, and returns the result.
* Return errors as text so the model can recover.

**Try it:** add a second tool `convert_currency(amount: float, from_code: str, to_code: str)` with its schema, add it to `TOOLS`, and test `run_tool` with good and bad arguments.